# ساخت نگاشت موجودیت‌ها (Entities) و روابط (Relations) به فرمت JSON

این نوت‌بوک دو کار انجام می‌دهد:

1. از روی فایل `entity.csv` یک فایل JSON با فرمت `{"m.<freebase_id>": "label, description"}` می‌سازد.
2. از روی فایل `relation.txt` یک فایل JSON با فرمت `{"relation_name": "relation_name"}` می‌سازد (هر رابطه به مقدار خودش نگاشت می‌شود).

**نکته:** مسیر فایل‌های ورودی و خروجی را در سلول تنظیمات زیر مطابق نیاز خودتان تغییر دهید.

In [3]:
import pandas as pd
import json
import os

# ---------------------- تنظیمات مسیرها ----------------------
ENTITY_CSV_PATH   = "entities_data.csv"              # مسیر فایل ورودی موجودیت‌ها
RELATION_TXT_PATH = "relations_integrated.txt"            # مسیر فایل ورودی روابط

ENTITY_JSON_OUT   = "entity_text.json"     # خروجی نگاشت موجودیت‌ها
RELATION_JSON_OUT = "relation_text.json"   # خروجی نگاشت روابط


## ۱. ساخت نگاشت موجودیت‌ها از `entity.csv`

In [4]:
# فایل با تب (\t) از هم جدا شده است
df = pd.read_csv(ENTITY_CSV_PATH, sep=",", dtype=str, keep_default_na=False)

print(df.shape)
df.head()


(10314, 9)


,freebase_id,wikidata_id,label,description,has_page,has_data,images,image_count,url
0,10016,Q128306,Denton,"city in Denton County, Texas, United States",TRUE,TRUE,caption=Old Courthouse Denton TX.jpg ; commons...,1,https://www.wikidata.org/wiki/Q128306
1,0100mt,Q16562,El Paso,"city in and county seat of El Paso County, Tex...",TRUE,TRUE,caption=El Paso Skyline.jpg ; commons=https://...,1,https://www.wikidata.org/wiki/Q16562
2,0102t4,Q52996,Marshall,"small city and county seat of Harrison County,...",TRUE,TRUE,caption=Downtown Marshall Wiki (1 of 1).jpg ; ...,1,https://www.wikidata.org/wiki/Q52996
3,0104lr,Q128282,Beaumont,"city and county seat of Jefferson County, Stat...",TRUE,TRUE,caption=BeaumontTX.JPG ; commons=https://commo...,1,https://www.wikidata.org/wiki/Q128282
4,0105y2,Q49273,Lubbock,"county seat of Lubbock County, Texas, United S...",TRUE,TRUE,caption=LubbockSkyline2013.jpg ; commons=https...,1,https://www.wikidata.org/wiki/Q49273


In [5]:
# def make_key(freebase_id: str) -> str:
#     """
#     freebase_id در فایل csv به شکل '0100gqqp' است.
#     فرمت استاندارد Freebase mid به صورت 'm.0100gqqp' است،
#     پس در صورتی که پیشوند 'm.' وجود نداشته باشد، اضافه می‌شود.
#     """
#     freebase_id = str(freebase_id).strip()
#     if not freebase_id:
#         return freebase_id
#     if freebase_id.startswith("m.") or freebase_id.startswith("/m/"):
#         return freebase_id.replace("/m/", "m.")
#     return f"{freebase_id}"


def clean_text(text: str) -> str:
    """
    بعضی از متن‌ها در فایل csv به‌خاطر escape شدن کوتیشن‌های داخلی،
    به‌صورت "" (دو کوتیشن پشت‌سرهم) نوشته شده‌اند که باید به یک
    کوتیشن معمولی (") تبدیل شوند. مثال:
        'title character of ""Phineas and Ferb""'
        -> 'title character of "Phineas and Ferb"'
    """
    text = (text or "").strip()
    text = text.replace('""', '"')
    return text


def make_value(label: str, description: str) -> str:
    """
    مقدار value از اتصال label و description ساخته می‌شود.
    اگر هرکدام خالی بودند، آن قسمت حذف می‌شود.
    اگر هر دو خالی بودند، رشته خالی برگردانده می‌شود.
    """
    label = clean_text(label)
    description = clean_text(description)
    
    parts = [p for p in (label, description) if p]
    if not parts:
        return ""
    return ", ".join(parts)


entity_map = {}
for _, row in df.iterrows():
    key = row["freebase_id"]
    if not key:
        continue
    value = make_value(row.get("label", ""), row.get("description", ""))
    entity_map[key] = value

print(f"تعداد موجودیت‌های ساخته‌شده: {len(entity_map)}")
# نمایش چند نمونه
list(entity_map.items())[:5]


تعداد موجودیت‌های ساخته‌شده: 9963


[('10016', 'Denton, city in Denton County, Texas, United States'),
 ('0100mt',
  'El Paso, city in and county seat of El Paso County, Texas, United States'),
 ('0102t4',
  'Marshall, small city and county seat of Harrison County, Texas, United States'),
 ('0104lr',
  'Beaumont, city and county seat of Jefferson County, State of Texas, United States'),
 ('0105y2', 'Lubbock, county seat of Lubbock County, Texas, United States')]

In [6]:
with open(ENTITY_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(entity_map, f, ensure_ascii=False, indent=2)

print(f"فایل ذخیره شد: {ENTITY_JSON_OUT}")


فایل ذخیره شد: entity_text.json


## ۲. ساخت نگاشت روابط از `relation.txt`

In [7]:
with open(RELATION_TXT_PATH, "r", encoding="utf-8") as f:
    relations = [line.strip() for line in f if line.strip()]

print(f"تعداد روابط: {len(relations)}")
relations[:5]


تعداد روابط: 91


['american_football*football_game',
 'american_football*football_game_0',
 'american_football*player_game_statistics',
 'american_football*player_passing_statistics',
 'american_football*player_receiving_statistics']

In [8]:
# هر رابطه به مقدار خودش نگاشت می‌شود
relation_map = {rel: rel for rel in relations}

with open(RELATION_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(relation_map, f, ensure_ascii=False, indent=2)

print(f"فایل ذخیره شد: {RELATION_JSON_OUT}")
list(relation_map.items())[:5]


فایل ذخیره شد: relation_text.json


[('american_football*football_game', 'american_football*football_game'),
 ('american_football*football_game_0', 'american_football*football_game_0'),
 ('american_football*player_game_statistics',
  'american_football*player_game_statistics'),
 ('american_football*player_passing_statistics',
  'american_football*player_passing_statistics'),
 ('american_football*player_receiving_statistics',
  'american_football*player_receiving_statistics')]

### توضیح ساختار خروجی‌ها

**entity_id2text.json**
```json
{
  "m.0100gqqp": "Grimm, season 4, season of television series",
  "m.01010z9z": "Bellator Fighting Championships, Bella Fighting Championships is a TV program."
}
```

**relation_id2text.json**
```json
{
  "american_football.player_receiving_statistics": "american_football.player_receiving_statistics",
  "architecture.occupancy": "architecture.occupancy"
}
```
